# Ingesting Markdown

In [ ]:
h1section = "# Required Filings and Due Dates\n\nThis paragraph is under the first H1 heading."
list_md = (
    "## SUTA Dumping Hurts Everyone\n\n"
    "Employers, employees, and taxpayers make up the difference in higher taxes, lost jobs, lost profits, lower wages, and higher costs for goods and services.\n\n"
    "SUTA dumping:\n\n"
    "* Costs the UI trust fund millions of dollars each year. Sublist follows:\n\n"
    "    * Adversely affects tax rates for all employers.\n"
    "    * Creates inequity for compliant employers. [some link](https://edd.ca.gov/en/disability/pfl_claim_process/)\n"
    "* Eliminates the incentive for employers to avoid layoffs.\n"
    "    * Compromises the integrity of the UI system.\n\n"
    "### [SUTA Dumping Schemes](https://edd.ca.gov/en/payroll_taxes/suta_dumping/#collapse-2a82e068-3b29-4473-af7e-de9ea6961277)\n\n"
    "These schemes are meant to unlawfully lower an employer\u2019s UI tax rate. Employers should know about these schemes and their potential legal ramifications."
)
table_md = (
    "## Scheduled Events\n\n"
    "### Orientation\n\n"
    "**Career Center Orientation**\n"
    "| Location | Date/Time | Other Information |\n"
    "| --- | --- | --- |\n"
    "| Virtual | Friday | This workshop will ... |\n"
    "| Virtual | Friday | This workshop is for ... |\n"
    "| Virtual | Friday | Staff will conduct ... |\n"
)
first_paragraph = "This is the first paragraph with no heading."
last_paragraph ="This is the last paragraph. In markdown and html, we can't associate this under the first H1 heading."
markdown_document = "\n\n".join([first_paragraph, h1section, list_md, table_md, last_paragraph])
print(markdown_document)

In [4]:
from langchain_text_splitters import MarkdownHeaderTextSplitter

In [ ]:
headers_to_split_on = [
    ("#", "H1"),
    ("##", "H2"),
    ("###", "H3"),
    ("####", "H4"),
    ("#####", "H5"),
    ("######", "H6"),
]

markdown_splitter = MarkdownHeaderTextSplitter(headers_to_split_on)
md_header_splits = markdown_splitter.split_text(markdown_document)
md_header_splits

In [ ]:
for doc in md_header_splits:
    headings = doc.metadata
    text = doc.page_content
    print("== ", " > ".join(headings))
    print(text)
    print("\n\n")

* Sublist got lost!

## Try different splitters

In [ ]:
from langchain.text_splitter import RecursiveCharacterTextSplitter
splitter_args = { "chunk_size": 512 }
text_splitter = RecursiveCharacterTextSplitter(
    # separators=["\n\n"],
    # Set a really small chunk size, just to show.
    **splitter_args
    # chunk_size = 512,
    # chunk_overlap  = 20
)
docs = text_splitter.create_documents([markdown_document])
for doc in docs:
    print("\n\n==")
    print(doc.page_content)

In [ ]:
from langchain.text_splitter import MarkdownTextSplitter
text_splitter = MarkdownTextSplitter(**splitter_args)
splits = text_splitter.split_text(markdown_document)
for split in splits:
    print("\n\n==")
    print(split)


## Use mistletoe to parse markdown input

In [ ]:
!pip install mistletoe

In [ ]:
from mistletoe.ast_renderer import AstRenderer
from mistletoe import Document, HTMLRenderer
# Parse Markdown into an AST
doc = Document(markdown_document)

# Access AST elements
print(doc.children)  # Access heading level
# print(doc.children[1].children[0].content)  # Access paragraph content

# Render AST back to HTML
out = AstRenderer().render(doc)
print(out)

In [ ]:
from mistletoe.block_token import *
p :Paragraph = doc.children[2]
print(p)

In [ ]:
render = AstRenderer().render

In [ ]:
print(render(p))

In [ ]:
import mistletoe.block_token as block_token
block_token.reset_tokens()
block_token._token_types

In [ ]:
from mistletoe.markdown_renderer import MarkdownRenderer
block_token.reset_tokens()
renderer = MarkdownRenderer(normalize_whitespace=True)
renderer.render(p)

In [ ]:
block_token.reset_tokens()
print(len(block_token._token_types), block_token._token_types)
with MarkdownRenderer(normalize_whitespace=True):
    print(len(block_token._token_types), block_token._token_types)
    pass
print(len(block_token._token_types), block_token._token_types)
with MarkdownRenderer(normalize_whitespace=True):
    print(len(block_token._token_types), block_token._token_types)
    pass


In [ ]:
from contextlib import contextmanager
@contextmanager
def md_renderer_cntx():
    renderer=MarkdownRenderer(normalize_whitespace=True)
    with renderer:
        yield renderer

with md_renderer_cntx() as cntx:
    print(cntx)

In [ ]:
block_token.reset_tokens()
with MarkdownRenderer(normalize_whitespace=True) as renderer:
    # the parsing phase is currently tightly connected with initiation and closing of a renderer.
    # Therefore, you should never call Document(...) outside of a with ... as renderer block,
    # unless you know what you are doing.
    doc = Document(markdown_document)
    p :Paragraph = doc.children[2]
    print(renderer.render(p))

In [ ]:
for c in doc.children:
    print(c)


In [ ]:
for i, c in enumerate(doc.children):
    print(i,"| ", renderer.render(c))

In [ ]:
c = doc.children[12]
print(c)
print(renderer.render(c))

In [ ]:
c.children

In [ ]:
render = renderer.render
sublist1 = c.children[0]
sublist1.children

In [ ]:
print(render(sublist1))

In [ ]:
sublist2 = c.children[1]
sublist2.children

In [ ]:
print(render(sublist2))

In [ ]:
table: Table = doc.children[23]

In [ ]:
print(render(table))

In [ ]:
table.header

In [ ]:
table.__dict__

In [ ]:
from mistletoe.block_token import TableRow
row1: TableRow = table.children[0]

In [ ]:
row1.__dict__

Lists and Tables are recognized!

## Let's see how the input markdown is affected

In [ ]:
import json
with open("../src/ingestion/edd_scrapings.json") as edd:
    edd_scrapings = json.load(edd)

for item in edd_scrapings:
    print(item["url"])

In [32]:
from difflib import SequenceMatcher
from difflib import unified_diff
import re

In [33]:

def compare_in_and_out(item):
    with MarkdownRenderer(normalize_whitespace=False) as renderer:
        # the parsing phase is currently tightly connected with initiation and closing of a renderer.
        # Therefore, you should never call Document(...) outside of a with ... as renderer block,
        # unless you know what you are doing.
        content = item.get("main_content", item.get("main_primary"))
        doc = Document(content)
        out_md = renderer.render(doc)

    fixed_content = []
    content = re.sub(r"\n\n ([a-zA-Z\[])", r"\n\n\1", content.rstrip(), flags=re.MULTILINE)
    content = re.sub(r"\n\n\n\n", "\n\n", content.rstrip(), flags=re.MULTILINE)
    content = re.sub(r"\n\n\n", "\n\n", content.rstrip(), flags=re.MULTILINE)
    for line in content.splitlines():
        if line.startswith("|"):
            continue

        # fix = re.sub(r"^	\+ ", "    + ", line.rstrip())
        # fix = re.sub(r"^\t\t- ", "        - ", fix.rstrip())
        fix = re.sub(r"^\t\t\t", r"            ", line.rstrip())
        fix = re.sub(r"^\t\t", r"        ", fix.rstrip())
        fix = re.sub(r"^\t", r"    ", fix.rstrip())
        fix = re.sub(r"^\t([0-9].) ", r"    \1 ", fix.rstrip())
        fixed_content.append(fix)
    # fixed_content = [line.rstrip() for line in content.splitlines() if not line.startswith("|")]
    fixed_out = [line.rstrip() for line in out_md.splitlines() if not line.startswith("|")]

    seq_match = SequenceMatcher(None, fixed_content, fixed_out)
    ratio = seq_match.ratio()
    if ratio < .9:
        print(item["url"])
        print(ratio)

        diff = unified_diff(fixed_content, fixed_out, lineterm='')
        print('\n'.join(list(diff)))
    

`MarkdownRenderer` produces `out_md` that:
- removes extra newlines,
- fixes indent of subsequent lines in a list item,
- removes extra leading space in a paragraph,


In [ ]:
for item in edd_scrapings:
    diff = compare_in_and_out(item)

### Try markdown-it-py

In [ ]:
!pip install markdown-it-py

In [ ]:
from markdown_it import *
from markdown_it.tree import SyntaxTreeNode

md = MarkdownIt("commonmark").enable('table')
tokens = md.parse(markdown_document)
node = SyntaxTreeNode(tokens)
print(node.pretty(indent=2, show_text=True))

In [ ]:
for i,c in enumerate(node.children):
    print(i, "|", c)

In [ ]:
print(node.children[6].pretty(indent=2, show_text=True))

In [ ]:
list_node: SyntaxTreeNode = node.children[6]
list_node.to_tokens()

In [ ]:
table_node: SyntaxTreeNode = node.children[12]
table_node.to_tokens()

In [ ]:
print(table_node.pretty(indent=2, show_text=True))

May be more detail than we need.

Let's stick with mistletoe since it works with tables out of the box.

For markdown-it, we have to use a 'tables' extension and configure the renderer:

In [ ]:
from mdformat.renderer import MDRenderer
import mdformat_tables

renderer = MDRenderer()
options = {"parser_extension": [mdformat_tables]}
env = {}
output_markdown = renderer.render(table_node.to_tokens(), options, env)
print(output_markdown)

## Test nutree

In [ ]:
!pip install nutree

In [44]:
from nutree import Tree, Node

In [ ]:
tree = Tree("Store")

n = tree.add("Records")

n.add("Let It Be")
n.add("Get Yer Ya-Ya's Out!")

n = tree.add("Books")
n.add("The Little Prince")

tree.print()


In [ ]:
print(tree['Records'].format())

In [153]:
class Person:
    def __init__(self, name, age, guid):
        self.name = name
        self.age = age
        self.guid = guid

In [ ]:
alice = Person("Alice", age=23, guid="{123-456}")
tree.add(alice, data_id=alice.guid)

# Lookup nodes by object, data_id, name pattern, ...
assert isinstance(tree[alice.guid].data, Person)
tree.print()


In [ ]:
bill = Person("Bill", age=23, guid="{123-457}")
tree.add(bill)
tree[bill]

In [156]:
del tree[bill]

In [ ]:
tree.print()

In [158]:
from nutree import IterMethod

### Test tree traversal to be used for chunking

In [ ]:
for node in tree.iterator(method=IterMethod.POST_ORDER):
    print(node)

In [ ]:
for node in tree['Records'].iterator(method=IterMethod.POST_ORDER):
    print(node)

Traversing children first and in order works!

Traversals: https://nutree.readthedocs.io/en/latest/ug_search_and_navigate.html#traversal
- See `visit()` for even more control

## Build document tree from parsed markdown

In [1]:
import sys
sys.path.append("../")
from src.ingestion.markdown_utils import *

Creating custom MarkdownRenderer: <mistletoe.markdown_renderer.MarkdownRenderer object at 0x109ac8290>


In [2]:
h1section = "# Required Filings and Due Dates\n\nThis paragraph is under the first H1 heading."
list_md = (
    "## SUTA Dumping Hurts Everyone\n\n"
    "Employers, employees, and taxpayers make up the difference in higher taxes, lost jobs, lost profits, lower wages, and higher costs for goods and services.\n\n"
    "SUTA dumping:\n\n"
    "* Costs the UI trust fund millions of dollars each year. Sublist follows:\n\n"
    "    * Adversely affects tax rates for all employers.\n"
    "    * Creates inequity for compliant employers. [some link](https://edd.ca.gov/en/disability/pfl_claim_process/)\n"
    "* Eliminates the incentive for employers to avoid layoffs.\n"
    "    * Compromises the integrity of the UI system.\n\n"
    "### [SUTA Dumping Schemes](https://edd.ca.gov/en/payroll_taxes/suta_dumping/#collapse-2a82e068-3b29-4473-af7e-de9ea6961277)\n\n"
    "These schemes are meant to unlawfully lower an employer\u2019s UI tax rate. Employers should know about these schemes and their potential legal ramifications."
)
table_md = (
    "## Scheduled Events\n\n"
    "### Orientation\n\n"
    "**Career Center Orientation**\n"
    "| Location | Date/Time | Other Information |\n"
    "| --- | --- | --- |\n"
    "| Virtual | Friday | This workshop will ... |\n"
    "| Virtual | Friday | This workshop is for ... |\n"
    "| Virtual | Friday | Staff will conduct ... |\n"
)
first_paragraph = "This is the first paragraph with no heading."
last_paragraph ="This is the last paragraph. In markdown and html, we can't associate this under the first H1 heading."
markdown_document = "\n\n".join([first_paragraph, h1section, list_md, table_md, last_paragraph])

"This is the first paragraph with no heading.\n\n# Required Filings and Due Dates\n\nThis paragraph is under the first H1 heading.\n\n## SUTA Dumping Hurts Everyone\n\nEmployers, employees, and taxpayers make up the difference in higher taxes, lost jobs, lost profits, lower wages, and higher costs for goods and services.\n\nSUTA dumping:\n\n* Costs the UI trust fund millions of dollars each year. Sublist follows:\n\n    * Adversely affects tax rates for all employers.\n    * Creates inequity for compliant employers. [some link](https://edd.ca.gov/en/disability/pfl_claim_process/)\n* Eliminates the incentive for employers to avoid layoffs.\n    * Compromises the integrity of the UI system.\n\n### [SUTA Dumping Schemes](https://edd.ca.gov/en/payroll_taxes/suta_dumping/#collapse-2a82e068-3b29-4473-af7e-de9ea6961277)\n\nThese schemes are meant to unlawfully lower an employer’s UI tax rate. Employers should know about these schemes and their potential legal ramifications.\n\n## Scheduled Ev

In [3]:
md_doc = normalize_markdown(markdown_document)

"This is the first paragraph with no heading.\n\n# Required Filings and Due Dates\n\nThis paragraph is under the first H1 heading.\n\n## SUTA Dumping Hurts Everyone\n\nEmployers, employees, and taxpayers make up the difference in higher taxes, lost jobs, lost profits, lower wages, and higher costs for goods and services.\n\nSUTA dumping:\n\n* Costs the UI trust fund millions of dollars each year. Sublist follows:\n\n    * Adversely affects tax rates for all employers.\n    * Creates inequity for compliant employers. [some link](https://edd.ca.gov/en/disability/pfl_claim_process/)\n* Eliminates the incentive for employers to avoid layoffs.\n    * Compromises the integrity of the UI system.\n\n### [SUTA Dumping Schemes](https://edd.ca.gov/en/payroll_taxes/suta_dumping/#collapse-2a82e068-3b29-4473-af7e-de9ea6961277)\n\nThese schemes are meant to unlawfully lower an employer’s UI tax rate. Employers should know about these schemes and their potential legal ramifications.\n\n## Scheduled Ev

In [4]:
print(md_doc)

This is the first paragraph with no heading.

# Required Filings and Due Dates

This paragraph is under the first H1 heading.

## SUTA Dumping Hurts Everyone

Employers, employees, and taxpayers make up the difference in higher taxes, lost jobs, lost profits, lower wages, and higher costs for goods and services.

SUTA dumping:

* Costs the UI trust fund millions of dollars each year. Sublist follows:

    * Adversely affects tax rates for all employers.
    * Creates inequity for compliant employers. [some link](https://edd.ca.gov/en/disability/pfl_claim_process/)
* Eliminates the incentive for employers to avoid layoffs.
    * Compromises the integrity of the UI system.

### [SUTA Dumping Schemes](https://edd.ca.gov/en/payroll_taxes/suta_dumping/#collapse-2a82e068-3b29-4473-af7e-de9ea6961277)

These schemes are meant to unlawfully lower an employer’s UI tax rate. Employers should know about these schemes and their potential legal ramifications.

## Scheduled Events

### Orientation

*

In [5]:
import mistletoe
# input:
doc =mistletoe.Document(markdown_document)

<mistletoe.block_token.Document with 14 children line_number=1 at 0x109bfb290>

In [6]:
doc.__dict__

{'footnotes': {},
 'line_number': 1,
 '_children': [<mistletoe.block_token.Paragraph with 1 child line_number=1 at 0x109ce5550>,
  <mistletoe.block_token.Heading with 1 child line_number=3 level=1 at 0x109ce6840>,
  <mistletoe.block_token.Paragraph with 1 child line_number=5 at 0x109ce70e0>,
  <mistletoe.block_token.Heading with 1 child line_number=7 level=2 at 0x109ce7050>,
  <mistletoe.block_token.Paragraph with 1 child line_number=9 at 0x109ce71d0>,
  <mistletoe.block_token.Paragraph with 1 child line_number=11 at 0x109ce72f0>,
  <mistletoe.block_token.List with 2 children line_number=13 loose=True start=None at 0x109ce7410>,
  <mistletoe.block_token.Heading with 1 child line_number=20 level=3 at 0x109ce7440>,
  <mistletoe.block_token.Paragraph with 1 child line_number=22 at 0x109ce7c80>,
  <mistletoe.block_token.Heading with 1 child line_number=24 level=2 at 0x109ce7dd0>,
  <mistletoe.block_token.Heading with 1 child line_number=26 level=3 at 0x109ce7ef0>,
  <mistletoe.block_token.

In [7]:
# From AstRenderer
def get_ast(token):
    """
    Recursively unrolls token attributes into dictionaries (token.children
    into lists).

    Returns:
        a dictionary of token's attributes.
    """
    node = {}
    # Python 3.6 uses [ordered dicts] [1].
    # Put in 'type' entry first to make the final tree format somewhat
    # similar to [MDAST] [2].
    #
    #   [1]: https://docs.python.org/3/whatsnew/3.6.html
    #   [2]: https://github.com/syntax-tree/mdast
    node['type'] = token.__class__.__name__
    for attrname in ['content', 'footnotes']:
        if attrname in vars(token):
            node[attrname] = getattr(token, attrname)
    for attrname in token.repr_attributes:
        node[attrname] = getattr(token, attrname)
    if 'header' in vars(token):
        node['header'] = get_ast(getattr(token, 'header'))
    if token.children is not None:
        node['children'] = [get_ast(child) for child in token.children]
    return node

In [8]:
get_ast(doc)

{'type': 'Document',
 'footnotes': {},
 'line_number': 1,
 'children': [{'type': 'Paragraph',
   'line_number': 1,
   'children': [{'type': 'RawText',
     'content': 'This is the first paragraph with no heading.'}]},
  {'type': 'Heading',
   'line_number': 3,
   'level': 1,
   'children': [{'type': 'RawText',
     'content': 'Required Filings and Due Dates'}]},
  {'type': 'Paragraph',
   'line_number': 5,
   'children': [{'type': 'RawText',
     'content': 'This paragraph is under the first H1 heading.'}]},
  {'type': 'Heading',
   'line_number': 7,
   'level': 2,
   'children': [{'type': 'RawText',
     'content': 'SUTA Dumping Hurts Everyone'}]},
  {'type': 'Paragraph',
   'line_number': 9,
   'children': [{'type': 'RawText',
     'content': 'Employers, employees, and taxpayers make up the difference in higher taxes, lost jobs, lost profits, lower wages, and higher costs for goods and services.'}]},
  {'type': 'Paragraph',
   'line_number': 11,
   'children': [{'type': 'RawText', 'c

#### MyNode

In [9]:
from nutree import Tree
tree = create_markdown_tree(markdown_document)


Tree<'Markdown tree'>

In [10]:
print(tree.format())

Tree<'Markdown tree'>
╰── MyNode my.0: Document: 
    ├── MyNode my.1: Paragraph: This is the first paragraph with no heading.

    │   ╰── MyNode my.2: RawText: This is the first paragraph with no heading.
    ├── MyNode my.3: Heading: # Required Filings and Due Dates

    │   ╰── MyNode my.4: RawText: Required Filings and Due Dates
    ├── MyNode my.5: Paragraph: This paragraph is under the first H1 heading.

    │   ╰── MyNode my.6: RawText: This paragraph is under the first H1 heading.
    ├── MyNode my.7: Heading: ## SUTA Dumping Hurts Everyone

    │   ╰── MyNode my.8: RawText: SUTA Dumping Hurts Everyone
    ├── MyNode my.9: Paragraph: Employers, employees, and taxpayers make up the difference in higher taxes, lost jobs, lost profits, lower wages, and higher costs for goods and services.

    │   ╰── MyNode my.10: RawText: Employers, employees, and taxpayers make up the difference in higher taxes, lost jobs, lost profits,
    ├── MyNode my.11: Paragraph: SUTA dumping:

    │   ╰

In [11]:
from nutree import IterMethod
from collections import defaultdict
type_attribs = defaultdict(set)
parent = defaultdict(set)
children = defaultdict(set)
token_types = defaultdict(tuple)
sample = {}
for node in tree.iterator(method=IterMethod.POST_ORDER):
    print(node.data.__dict__)
    type_attribs[node.data.type].update(node.data.__dict__.keys())
    children[node.data.type].update([child.data.type for child in node.children])
    parent[node.data.type].add(node.data.token.parent.__class__.__name__)
    if not node.data.type in sample:
        sample[node.data.type] = node
    if not node.data.type in token_types:
        token_types[node.data.type] = node.data.token.__class__.__bases__

from pprint import pprint
pprint(type_attribs)
pprint(children)

{'id': 'my.2', 'token': <mistletoe.span_token.RawText content='This is the first paragraph wi'...+14 at 0x109ce7fb0>, 'type': 'RawText', 'chunked': False, 'summaries': [], 'heading_token': None, 'content': 'This is the first paragraph with no heading.'}
{'id': 'my.1', 'token': <mistletoe.block_token.Paragraph with 1 child line_number=1 at 0x109ce6450>, 'type': 'Paragraph', 'chunked': False, 'summaries': [], 'heading_token': None, 'line_number': 1}
{'id': 'my.4', 'token': <mistletoe.span_token.RawText content='Required Filings and Due Dates' at 0x109ce51c0>, 'type': 'RawText', 'chunked': False, 'summaries': [], 'heading_token': None, 'content': 'Required Filings and Due Dates'}
{'id': 'my.3', 'token': <mistletoe.block_token.Heading with 1 child line_number=3 level=1 at 0x109ce5af0>, 'type': 'Heading', 'chunked': False, 'summaries': [], 'heading_token': None, 'line_number': 3, 'level': 1}
{'id': 'my.6', 'token': <mistletoe.span_token.RawText content='This paragraph is under the fi'...+15

In [12]:
tree.print()

Tree<'Markdown tree'>
╰── MyNode my.0: Document: 
    ├── MyNode my.1: Paragraph: This is the first paragraph with no heading.

    │   ╰── MyNode my.2: RawText: This is the first paragraph with no heading.
    ├── MyNode my.3: Heading: # Required Filings and Due Dates

    │   ╰── MyNode my.4: RawText: Required Filings and Due Dates
    ├── MyNode my.5: Paragraph: This paragraph is under the first H1 heading.

    │   ╰── MyNode my.6: RawText: This paragraph is under the first H1 heading.
    ├── MyNode my.7: Heading: ## SUTA Dumping Hurts Everyone

    │   ╰── MyNode my.8: RawText: SUTA Dumping Hurts Everyone
    ├── MyNode my.9: Paragraph: Employers, employees, and taxpayers make up the difference in higher taxes, lost jobs, lost profits, lower wages, and higher costs for goods and services.

    │   ╰── MyNode my.10: RawText: Employers, employees, and taxpayers make up the difference in higher taxes, lost jobs, lost profits,
    ├── MyNode my.11: Paragraph: SUTA dumping:

    │   ╰

In [13]:
parent

defaultdict(set,
            {'RawText': {'Heading',
              'Link',
              'Paragraph',
              'Strong',
              'TableCell'},
             'Paragraph': {'Document', 'ListItem'},
             'Heading': {'Document'},
             'ListItem': {'List'},
             'Link': {'Heading', 'Paragraph'},
             'List': {'Document', 'ListItem'},
             'Strong': {'Paragraph'},
             'TableCell': {'TableRow'},
             'TableRow': {'NoneType', 'Table'},
             'Table': {'Document'},
             'Document': {'NoneType'}})

In [14]:
sample

{'RawText': Node<'MyNode my.2: RawText: This is the first paragraph with no heading.', data_id=my.2>,
 'Paragraph': Node<'MyNode my.1: Paragraph: This is the first paragraph with no heading.\n', data_id=my.1>,
 'Heading': Node<'MyNode my.3: Heading: # Required Filings and Due Dates\n', data_id=my.3>,
 'ListItem': Node<'MyNode my.18: ListItem:   * Adversely affects tax rates for all employers.\n', data_id=my.18>,
 'Link': Node<'MyNode my.24: Link: ', data_id=my.24>,
 'List': Node<'MyNode my.17: List: ', data_id=my.17>,
 'Strong': Node<'MyNode my.43: Strong: ', data_id=my.43>,
 'TableCell': Node<'MyNode my.47: TableCell: ', data_id=my.47>,
 'TableRow': Node<'MyNode my.46: TableRow: | Location | Date/Time | Other Information |\n', data_id=my.46>,
 'Table': Node<'MyNode my.45: Table: ', data_id=my.45>,
 'Document': Node<'MyNode my.0: Document: ', data_id=my.0>}

In [15]:
sample['Strong'].data.__dict__

{'id': 'my.43',
 'token': <mistletoe.span_token.Strong with 1 child at 0x109cf0980>,
 'type': 'Strong',
 'chunked': False,
 'summaries': [],
 'heading_token': None}

In [16]:
from mistletoe.markdown_renderer import MarkdownRenderer

# Set max_line_length so that TableRow can be rendered
renderer = create_md_renderer()
def render(*args):
    print(renderer.render(*args))

Creating custom MarkdownRenderer: <mistletoe.markdown_renderer.MarkdownRenderer object at 0x109ce5ca0>


In [17]:
render(sample['Strong'].data.token)

**Career Center Orientation**



In [18]:
render(sample['ListItem'].data.token)

  * Adversely affects tax rates for all employers.



In [19]:
render(sample['Table'].data.token)

| Location | Date/Time | Other Information        |
| -------- | --------- | ------------------------ |
| Virtual  | Friday    | This workshop will ...   |
| Virtual  | Friday    | This workshop is for ... |
| Virtual  | Friday    | Staff will conduct ...   |



In [20]:
token_types

defaultdict(tuple,
            {'RawText': (mistletoe.span_token.SpanToken,),
             'Paragraph': (mistletoe.block_token.BlockToken,),
             'Heading': (mistletoe.block_token.BlockToken,),
             'ListItem': (mistletoe.block_token.BlockToken,),
             'Link': (mistletoe.span_token.SpanToken,),
             'List': (mistletoe.block_token.BlockToken,),
             'Strong': (mistletoe.span_token.SpanToken,),
             'TableCell': (mistletoe.block_token.BlockToken,),
             'TableRow': (mistletoe.block_token.BlockToken,),
             'Table': (mistletoe.block_token.BlockToken,),
             'Document': (mistletoe.block_token.BlockToken,)})

In [21]:
block_token.List

mistletoe.block_token.List

In [22]:
from mistletoe import span_token
span_token.Link

mistletoe.span_token.Link

### Collapse SpanTokens into parent node

In [23]:
sample['TableRow'].data

MyNode my.46: TableRow: | Location | Date/Time | Other Information |

In [24]:
sample['TableRow'].parent

Node<'MyNode my.45: Table: ', data_id=my.45>

In [25]:
sample['TableRow'].data.__dict__

{'id': 'my.46',
 'token': <mistletoe.block_token.TableRow with 3 children line_number=29 row_align=[None, None, None] at 0x109cf0b00>,
 'type': 'TableRow',
 'chunked': False,
 'summaries': [],
 'heading_token': None,
 'line_number': 29,
 'row_align': [None, None, None]}

In [26]:
sample['Table'].data.__dict__

{'id': 'my.45',
 'token': <mistletoe.block_token.Table with 3 children line_number=29 column_align=[None, None, None] at 0x109cf0e60>,
 'type': 'Table',
 'chunked': False,
 'summaries': [],
 'heading_token': None,
 'line_number': 29,
 'column_align': [None, None, None],
 'header': Node<'MyNode my.46: TableRow: | Location | Date/Time | Other Information |\n', data_id=my.46>}

In [27]:
# Header TableRow doesn't have a parent!
sample['TableRow'].data.token.parent

In [28]:
sample['Table'].data.token

<mistletoe.block_token.Table with 3 children line_number=29 column_align=[None, None, None] at 0x109cf0e60>

In [29]:
sample['Table'].data.token.children

[<mistletoe.block_token.TableRow with 3 children line_number=31 row_align=[None, None, None] at 0x109cf0710>,
 <mistletoe.block_token.TableRow with 3 children line_number=32 row_align=[None, None, None] at 0x109cf1130>,
 <mistletoe.block_token.TableRow with 3 children line_number=33 row_align=[None, None, None] at 0x109cf1400>]

In [30]:
for c in sample['Table'].data.token.children:
    render(c)

| Virtual | Friday | This workshop will ... |

| Virtual | Friday | This workshop is for ... |

| Virtual | Friday | Staff will conduct ... |



In [31]:
render(sample['Table'].data.token.header)

| Location | Date/Time | Other Information |



In [32]:
header=sample['Table'].data.token.header
for c in header.children:
    print(c.children[0].content)
[c.children[0].content for c in header.children]

Location
Date/Time
Other Information


['Location', 'Date/Time', 'Other Information']

In [33]:
render(sample['TableRow'].data.token)

| Location | Date/Time | Other Information |



In [34]:
block_token.TableRow

mistletoe.block_token.TableRow

In [35]:
from nutree import IterMethod

In [36]:
tree2 = create_markdown_tree(markdown_document, normalize_md=False)
print("==== Tree created ====")
tree2.print()

==== Tree created ====
Tree<'Markdown tree'>
╰── MyNode my.76: Document: 
    ├── MyNode my.77: Paragraph: This is the first paragraph with no heading.

    │   ╰── MyNode my.78: RawText: This is the first paragraph with no heading.
    ├── MyNode my.79: Heading: # Required Filings and Due Dates

    │   ╰── MyNode my.80: RawText: Required Filings and Due Dates
    ├── MyNode my.81: Paragraph: This paragraph is under the first H1 heading.

    │   ╰── MyNode my.82: RawText: This paragraph is under the first H1 heading.
    ├── MyNode my.83: Heading: ## SUTA Dumping Hurts Everyone

    │   ╰── MyNode my.84: RawText: SUTA Dumping Hurts Everyone
    ├── MyNode my.85: Paragraph: Employers, employees, and taxpayers make up the difference in higher taxes, lost jobs, lost profits, lower wages, and higher costs for goods and services.

    │   ╰── MyNode my.86: RawText: Employers, employees, and taxpayers make up the difference in higher taxes, lost jobs, lost profits,
    ├── MyNode my.87: Pa

In [37]:
from nutree import diff_node_formatter

tree = create_markdown_tree(markdown_document, normalize_md=True)
tree2 = create_markdown_tree(markdown_document, normalize_md=False)
tree_diff = tree.diff(tree2)
tree_diff.print(repr=diff_node_formatter)

Tree<"diff('Markdown tree', 'Markdown tree')">
├── MyNode my.152: Document:  - [Removed]
╰── MyNode my.228: Document:  - [Added]
    ├── MyNode my.229: Paragraph: This is the first paragraph with no heading.
 - [Added]
    │   ╰── MyNode my.230: RawText: This is the first paragraph with no heading.
    ├── MyNode my.231: Heading: # Required Filings and Due Dates
 - [Added]
    │   ╰── MyNode my.232: RawText: Required Filings and Due Dates
    ├── MyNode my.233: Paragraph: This paragraph is under the first H1 heading.
 - [Added]
    │   ╰── MyNode my.234: RawText: This paragraph is under the first H1 heading.
    ├── MyNode my.235: Heading: ## SUTA Dumping Hurts Everyone
 - [Added]
    │   ╰── MyNode my.236: RawText: SUTA Dumping Hurts Everyone
    ├── MyNode my.237: Paragraph: Employers, employees, and taxpayers make up the difference in higher taxes, lost jobs, lost profits, lower wages, and higher costs for goods and services.
 - [Added]
    │   ╰── MyNode my.238: RawText: Employers,

In [38]:
tree = create_markdown_tree(markdown_document, normalize_md=True)
print("==== Tree created ====")
tree.print()
rendering = collapse_span_tokens(tree)
print("\n".join(rendering))

==== Tree created ====
Tree<'Markdown tree'>
╰── MyNode my.304: Document: 
    ├── MyNode my.305: Paragraph: This is the first paragraph with no heading.

    │   ╰── MyNode my.306: RawText: This is the first paragraph with no heading.
    ├── MyNode my.307: Heading: # Required Filings and Due Dates

    │   ╰── MyNode my.308: RawText: Required Filings and Due Dates
    ├── MyNode my.309: Paragraph: This paragraph is under the first H1 heading.

    │   ╰── MyNode my.310: RawText: This paragraph is under the first H1 heading.
    ├── MyNode my.311: Heading: ## SUTA Dumping Hurts Everyone

    │   ╰── MyNode my.312: RawText: SUTA Dumping Hurts Everyone
    ├── MyNode my.313: Paragraph: Employers, employees, and taxpayers make up the difference in higher taxes, lost jobs, lost profits, lower wages, and higher costs for goods and services.

    │   ╰── MyNode my.314: RawText: Employers, employees, and taxpayers make up the difference in higher taxes, lost jobs, lost profits,
    ├── MyNod

In [39]:
render(doc)

This is the first paragraph with no heading.
# Required Filings and Due Dates
This paragraph is under the first H1 heading.
## SUTA Dumping Hurts Everyone
Employers, employees, and taxpayers make up the difference in higher taxes, lost jobs, lost profits, lower wages, and higher costs for goods and services.
SUTA dumping:
* Costs the UI trust fund millions of dollars each year. Sublist follows:
    * Adversely affects tax rates for all employers.
    * Creates inequity for compliant employers. [some link](https://edd.ca.gov/en/disability/pfl_claim_process/)
* Eliminates the incentive for employers to avoid layoffs.
    * Compromises the integrity of the UI system.
### [SUTA Dumping Schemes](https://edd.ca.gov/en/payroll_taxes/suta_dumping/#collapse-2a82e068-3b29-4473-af7e-de9ea6961277)
These schemes are meant to unlawfully lower an employer’s UI tax rate. Employers should know about these schemes and their potential legal ramifications.
## Scheduled Events
### Orientation
**Career Cent

In [40]:
tree.print()

Tree<'Markdown tree'>
╰── MyNode my.304: Document: 
    ├── MyNode my.305: Paragraph: This is the first paragraph with no heading.

    ├── MyNode my.307: Heading: # Required Filings and Due Dates

    ├── MyNode my.309: Paragraph: This paragraph is under the first H1 heading.

    ├── MyNode my.311: Heading: ## SUTA Dumping Hurts Everyone

    ├── MyNode my.313: Paragraph: Employers, employees, and taxpayers make up the difference in higher taxes, lost jobs, lost profits, lower wages, and higher costs for goods and services.

    ├── MyNode my.315: Paragraph: SUTA dumping:

    ├── MyNode my.317: List: 
    │   ├── MyNode my.318: ListItem: * Costs the UI trust fund millions of dollars each year. Sublist follows:
    * Adversely affects tax rates for all employers.
    * Creates inequity for compliant employers. [some link](https://edd.ca.gov/en/disability/pfl_claim_process/)

    │   │   ├── MyNode my.319: Paragraph: Costs the UI trust fund millions of dollars each year. Sublist follo

### Move nodes under associated heading nodes

In [41]:
# Find all headings

create_heading_sections(tree)
print("==== Tree updated ====")
tree.print()

MyNode my.307: Heading: # Required Filings and Due Dates

>  MyNode my.309: Paragraph: This paragraph is under the first H1 heading.

MyNode my.311: Heading: ## SUTA Dumping Hurts Everyone

>  MyNode my.313: Paragraph: Employers, employees, and taxpayers make up the difference in higher taxes, lost jobs, lost profits, lower wages, and higher costs for goods and services.

>  MyNode my.315: Paragraph: SUTA dumping:

>  MyNode my.317: List: 
MyNode my.337: Heading: ### [SUTA Dumping Schemes](https://edd.ca.gov/en/payroll_taxes/suta_dumping/#collapse-2a82e068-3b29-4473-af7e-de9ea6961277)

>  MyNode my.340: Paragraph: These schemes are meant to unlawfully lower an employer’s UI tax rate. Employers should know about these schemes and their potential legal ramifications.

MyNode my.342: Heading: ## Scheduled Events

MyNode my.344: Heading: ### Orientation

>  MyNode my.346: Paragraph: **Career Center Orientation**

>  MyNode my.349: Table: 
>  MyNode my.378: Paragraph: This is the last parag

In [42]:
render(doc)

This is the first paragraph with no heading.
# Required Filings and Due Dates
This paragraph is under the first H1 heading.
## SUTA Dumping Hurts Everyone
Employers, employees, and taxpayers make up the difference in higher taxes, lost jobs, lost profits, lower wages, and higher costs for goods and services.
SUTA dumping:
* Costs the UI trust fund millions of dollars each year. Sublist follows:
    * Adversely affects tax rates for all employers.
    * Creates inequity for compliant employers. [some link](https://edd.ca.gov/en/disability/pfl_claim_process/)
* Eliminates the incentive for employers to avoid layoffs.
    * Compromises the integrity of the UI system.
### [SUTA Dumping Schemes](https://edd.ca.gov/en/payroll_taxes/suta_dumping/#collapse-2a82e068-3b29-4473-af7e-de9ea6961277)
These schemes are meant to unlawfully lower an employer’s UI tax rate. Employers should know about these schemes and their potential legal ramifications.
## Scheduled Events
### Orientation
**Career Cent

## Char-based chunking

In [43]:
token_types

defaultdict(tuple,
            {'RawText': (mistletoe.span_token.SpanToken,),
             'Paragraph': (mistletoe.block_token.BlockToken,),
             'Heading': (mistletoe.block_token.BlockToken,),
             'ListItem': (mistletoe.block_token.BlockToken,),
             'Link': (mistletoe.span_token.SpanToken,),
             'List': (mistletoe.block_token.BlockToken,),
             'Strong': (mistletoe.span_token.SpanToken,),
             'TableCell': (mistletoe.block_token.BlockToken,),
             'TableRow': (mistletoe.block_token.BlockToken,),
             'Table': (mistletoe.block_token.BlockToken,),
             'Document': (mistletoe.block_token.BlockToken,)})

In [44]:
def check_nodes(node: Node, memo):
    print(isinstance(node.data.token, block_token.BlockToken), node.data)
tree.visit(check_nodes)

True MyNode my.304: Document: 
True MyNode my.305: Paragraph: This is the first paragraph with no heading.

False MyNode my.380: HeadingSection: # Required Filings and Due Dates

True MyNode my.307: Heading: # Required Filings and Due Dates

True MyNode my.309: Paragraph: This paragraph is under the first H1 heading.

False MyNode my.381: HeadingSection: ## SUTA Dumping Hurts Everyone

True MyNode my.311: Heading: ## SUTA Dumping Hurts Everyone

True MyNode my.313: Paragraph: Employers, employees, and taxpayers make up the difference in higher taxes, lost jobs, lost profits, lower wages, and higher costs for goods and services.

True MyNode my.315: Paragraph: SUTA dumping:

True MyNode my.317: List: 
True MyNode my.318: ListItem: * Costs the UI trust fund millions of dollars each year. Sublist follows:
    * Adversely affects tax rates for all employers.
    * Creates inequity for compliant employers. [some link](https://edd.ca.gov/en/disability/pfl_claim_process/)

True MyNode my.319:

In [45]:
render(doc)

This is the first paragraph with no heading.
# Required Filings and Due Dates
This paragraph is under the first H1 heading.
## SUTA Dumping Hurts Everyone
Employers, employees, and taxpayers make up the difference in higher taxes, lost jobs, lost profits, lower wages, and higher costs for goods and services.
SUTA dumping:
* Costs the UI trust fund millions of dollars each year. Sublist follows:
    * Adversely affects tax rates for all employers.
    * Creates inequity for compliant employers. [some link](https://edd.ca.gov/en/disability/pfl_claim_process/)
* Eliminates the incentive for employers to avoid layoffs.
    * Compromises the integrity of the UI system.
### [SUTA Dumping Schemes](https://edd.ca.gov/en/payroll_taxes/suta_dumping/#collapse-2a82e068-3b29-4473-af7e-de9ea6961277)
These schemes are meant to unlawfully lower an employer’s UI tax rate. Employers should know about these schemes and their potential legal ramifications.
## Scheduled Events
### Orientation
**Career Cent

### Simple

In [46]:
from nutree import SkipBranch

def chunk_nodes(node: Node, memo):
    md_str = render_node_as_markdown(node)
    if len(md_str) < CHAR_LIMIT:
        print(f"Chunked {node.data.type} ({node.data.id}) with len {len(md_str)}")
        chunking_memo[node.data.id].append(md_str)
        # Prevent visiting the child nodes:
        return SkipBranch
    else:
        print(f"?? Chunking {node.data.type} with {len(md_str)}")

chunking_memo = defaultdict(list)
tree.visit(chunk_nodes, memo=chunking_memo)
pprint(chunking_memo)

True Document
?? Chunking Document with 1379
True Paragraph
Chunked Paragraph (my.305) with len 45
False HeadingSection
Chunked HeadingSection (my.380) with len 80
False HeadingSection
?? Chunking HeadingSection with 552
True Heading
Chunked Heading (my.311) with len 31
True Paragraph
Chunked Paragraph (my.313) with len 155
True Paragraph
Chunked Paragraph (my.315) with len 14
True List
Chunked List (my.317) with len 349
False HeadingSection
Chunked HeadingSection (my.382) with len 279
True Heading
Chunked Heading (my.342) with len 20
False HeadingSection
Chunked HeadingSection (my.383) with len 411
defaultdict(<class 'list'>,
            {'my.305': ['This is the first paragraph with no heading.\n'],
             'my.311': ['## SUTA Dumping Hurts Everyone\n'],
             'my.313': ['Employers, employees, and taxpayers make up the '
                        'difference in higher taxes, lost jobs, lost profits, '
                        'lower wages, and higher costs for goods and '
   

### Simple Recursive

In [47]:
tree.print()

Tree<'Markdown tree'>
╰── MyNode my.304: Document: 
    ├── MyNode my.305: Paragraph: This is the first paragraph with no heading.

    ├── MyNode my.380: HeadingSection: # Required Filings and Due Dates

    │   ├── MyNode my.307: Heading: # Required Filings and Due Dates

    │   ╰── MyNode my.309: Paragraph: This paragraph is under the first H1 heading.

    ├── MyNode my.381: HeadingSection: ## SUTA Dumping Hurts Everyone

    │   ├── MyNode my.311: Heading: ## SUTA Dumping Hurts Everyone

    │   ├── MyNode my.313: Paragraph: Employers, employees, and taxpayers make up the difference in higher taxes, lost jobs, lost profits, lower wages, and higher costs for goods and services.

    │   ├── MyNode my.315: Paragraph: SUTA dumping:

    │   ╰── MyNode my.317: List: 
    │       ├── MyNode my.318: ListItem: * Costs the UI trust fund millions of dollars each year. Sublist follows:
    * Adversely affects tax rates for all employers.
    * Creates inequity for compliant employers. [som

In [48]:
CHAR_LIMIT = 500

chunking_memo = defaultdict(list)
for c in tree.children:
    chunk_nodes(c, memo=chunking_memo)
pprint(chunking_memo, width=120)

True Document
?? Chunking Document with 1379
defaultdict(<class 'list'>, {})


In [49]:
for c in tree.children[0].children:
    print(c)

Node<'MyNode my.305: Paragraph: This is the first paragraph with no heading.\n', data_id=my.305>
Node<'MyNode my.380: HeadingSection: # Required Filings and Due Dates\n', data_id=278703033>
Node<'MyNode my.381: HeadingSection: ## SUTA Dumping Hurts Everyone\n', data_id=278733126>
Node<'MyNode my.382: HeadingSection: ### [SUTA Dumping Schemes](https://edd.ca.gov/en/payroll_taxes/suta_dumping/#collapse-2a82e068-3b29-4473-af7e-de9ea6961277)\n', data_id=278733633>
Node<'MyNode my.342: Heading: ## Scheduled Events\n', data_id=my.342>
Node<'MyNode my.383: HeadingSection: ### Orientation\n', data_id=278733225>


In [50]:
tree.print()

Tree<'Markdown tree'>
╰── MyNode my.304: Document: 
    ├── MyNode my.305: Paragraph: This is the first paragraph with no heading.

    ├── MyNode my.380: HeadingSection: # Required Filings and Due Dates

    │   ├── MyNode my.307: Heading: # Required Filings and Due Dates

    │   ╰── MyNode my.309: Paragraph: This paragraph is under the first H1 heading.

    ├── MyNode my.381: HeadingSection: ## SUTA Dumping Hurts Everyone

    │   ├── MyNode my.311: Heading: ## SUTA Dumping Hurts Everyone

    │   ├── MyNode my.313: Paragraph: Employers, employees, and taxpayers make up the difference in higher taxes, lost jobs, lost profits, lower wages, and higher costs for goods and services.

    │   ├── MyNode my.315: Paragraph: SUTA dumping:

    │   ╰── MyNode my.317: List: 
    │       ├── MyNode my.318: ListItem: * Costs the UI trust fund millions of dollars each year. Sublist follows:
    * Adversely affects tax rates for all employers.
    * Creates inequity for compliant employers. [som

# Other

## Ideas

* Use each webpage JSON object fields to determine which parsing+chunking strategy to use
* Parsing strategies:
  - convert to MD
  - fix/adjust MD; normalize MD
  - tree prep, eg: collapse block-token tree nodes, move nodes under associated heading node, ...
* Chunking strategies: 
  - traverse tree
    - from root
    - from leaves
    - ...
  - (create ipynb to demo the process)
  - main (configurable) operations:
    - summarize/shorten a node
    - create chunk
    - within_chunk_limit(node)

In [1]:
import sys
sys.path.append("../")
from src.ingestion.markdown_utils import *

In [2]:
h1section = "# Required Filings and Due Dates\n\nThis paragraph is under the first H1 heading."
list_md = (
    "### SUTA Dumping Hurts Everyone\n\n"
    "* List immediate after heading\n\n"
    "Employers, employees, and taxpayers make up the difference in higher taxes, lost jobs, lost profits, lower wages, and higher costs for goods and services.\n\n"
    "SUTA dumping:\n\n"
    "* Costs the UI trust fund millions of dollars each year. Sublist follows:\n\n"
    "    * Adversely affects tax rates for all employers.\n"
    "    * Creates inequity for compliant employers. [some link](https://edd.ca.gov/en/disability/pfl_claim_process/)\n"
    "* Eliminates the incentive for employers to avoid layoffs.\n"
    "    * Compromises the integrity of the UI system.\n\n"
    "### [SUTA Dumping Schemes](https://edd.ca.gov/en/payroll_taxes/suta_dumping/#collapse-2a82e068-3b29-4473-af7e-de9ea6961277)\n\n"
    "These schemes are meant to unlawfully lower an employer\u2019s UI tax rate. Employers should know about these schemes and their potential legal ramifications."
)
table_md = (
    "## Scheduled Events\n\n"
    "### Orientation\n\n"
    "**Career Center Orientation**\n"
    "| Location | Date/Time | Other Information |\n"
    "| --- | --- | --- |\n"
    "| Virtual | Friday | This workshop will ... |\n"
    "| Virtual | Fri | This workshop is for ... |\n"
    "| Virtual | F | Staff will conduct ... |\n"
)
first_paragraph = "This is the first paragraph with no heading."
last_paragraph ="This is the last paragraph. In markdown and html, we can't associate this under the first H1 heading."
markdown_document = "\n\n".join([first_paragraph, h1section, list_md, table_md, last_paragraph])

"This is the first paragraph with no heading.\n\n# Required Filings and Due Dates\n\nThis paragraph is under the first H1 heading.\n\n### SUTA Dumping Hurts Everyone\n\n* List immediate after heading\n\nEmployers, employees, and taxpayers make up the difference in higher taxes, lost jobs, lost profits, lower wages, and higher costs for goods and services.\n\nSUTA dumping:\n\n* Costs the UI trust fund millions of dollars each year. Sublist follows:\n\n    * Adversely affects tax rates for all employers.\n    * Creates inequity for compliant employers. [some link](https://edd.ca.gov/en/disability/pfl_claim_process/)\n* Eliminates the incentive for employers to avoid layoffs.\n    * Compromises the integrity of the UI system.\n\n### [SUTA Dumping Schemes](https://edd.ca.gov/en/payroll_taxes/suta_dumping/#collapse-2a82e068-3b29-4473-af7e-de9ea6961277)\n\nThese schemes are meant to unlawfully lower an employer’s UI tax rate. Employers should know about these schemes and their potential lega

In [3]:
tree = create_markdown_tree(markdown_document, normalize_md=True)

Tree<'Markdown tree'>

In [4]:
print(markdown_document)

This is the first paragraph with no heading.

# Required Filings and Due Dates

This paragraph is under the first H1 heading.

### SUTA Dumping Hurts Everyone

* List immediate after heading

Employers, employees, and taxpayers make up the difference in higher taxes, lost jobs, lost profits, lower wages, and higher costs for goods and services.

SUTA dumping:

* Costs the UI trust fund millions of dollars each year. Sublist follows:

    * Adversely affects tax rates for all employers.
    * Creates inequity for compliant employers. [some link](https://edd.ca.gov/en/disability/pfl_claim_process/)
* Eliminates the incentive for employers to avoid layoffs.
    * Compromises the integrity of the UI system.

### [SUTA Dumping Schemes](https://edd.ca.gov/en/payroll_taxes/suta_dumping/#collapse-2a82e068-3b29-4473-af7e-de9ea6961277)

These schemes are meant to unlawfully lower an employer’s UI tax rate. Employers should know about these schemes and their potential legal ramifications.

## Sch

In [5]:
print(normalize_markdown(normalize_markdown(markdown_document)))

This is the first paragraph with no heading.

# Required Filings and Due Dates

This paragraph is under the first H1 heading.

### SUTA Dumping Hurts Everyone

* List immediate after heading

Employers, employees, and taxpayers make up the difference in higher taxes, lost jobs, lost profits, lower wages, and higher costs for goods and services.

SUTA dumping:

* Costs the UI trust fund millions of dollars each year. Sublist follows:

    * Adversely affects tax rates for all employers.
    * Creates inequity for compliant employers. [some link](https://edd.ca.gov/en/disability/pfl_claim_process/)
* Eliminates the incentive for employers to avoid layoffs.
    * Compromises the integrity of the UI system.

### [SUTA Dumping Schemes](https://edd.ca.gov/en/payroll_taxes/suta_dumping/#collapse-2a82e068-3b29-4473-af7e-de9ea6961277)

These schemes are meant to unlawfully lower an employer’s UI tax rate. Employers should know about these schemes and their potential legal ramifications.

## Sch

In [6]:
# block_token.reset_tokens()
# with MarkdownRenderer(normalize_whitespace=False) as renderer:
#     # "the parsing phase is currently tightly connected with initiation and closing of a renderer.
#     # Therefore, you should never call Document(...) outside of a with ... as renderer block"
#     doc = mistletoe.Document(markdown2)
#     print(renderer.render(doc))

In [7]:
# for node in tree.iterator(method=IterMethod.PRE_ORDER):
#     print(node.data.render())
#     print("")

In [8]:
tree.print()

RENDERING Heading (H1.3)
RENDERING Heading (H3.7)
RENDERING Link (s.10)
RENDERING Heading (H3.22)
RENDERING Link (s.14)
RENDERING Heading (H2.26)
RENDERING Heading (H3.28)
RENDERING TableRow (TR33)
RENDERING TableRow (TR34)
RENDERING TableRow (TR35)
Tree<'Markdown tree'>
╰── Document (D1): '<mistletoe.block_token.Document with 15 children line_number=1 at 0x103fb2360>'
    ├── Paragraph (P1) of length 45 across 1 children
    │   ╰── RawText (s.0): "<mistletoe.span_token.RawText content='This is the first paragraph wi'...+14 at 0x104038c50>"
    ├── Heading (H1.3): '# Required Filings and Due Dates\n'
    │   ╰── RawText (s.1): "<mistletoe.span_token.RawText content='Required Filings and Due Dates' at 0x10403bec0>"
    ├── Paragraph (P5) of length 46 across 1 children
    │   ╰── RawText (s.2): "<mistletoe.span_token.RawText content='This paragraph is under the fi'...+15 at 0x10403be00>"
    ├── Heading (H3.7): '### SUTA Dumping Hurts Everyone\n'
    │   ╰── RawText (s.3): "<mistletoe.

In [9]:
rendering = collapse_span_tokens(tree)
create_heading_sections(tree)

5

In [10]:
tree.print()

Tree<'Markdown tree'>
╰── Document (D1): '<mistletoe.block_token.Document with 15 children line_number=1 at 0x103fb2360>'
    ├── Paragraph (P1) of length 45 across 1 children: 'This is the first paragraph with no heading.'
    ├── HeadingSection (_H1.1) with 2 children
    │   ├── Heading (H1.3): '# Required Filings and Due Dates'
    │   ╰── Paragraph (P5) of length 46 across 1 children: 'This paragraph is under the first H1 heading.'
    ├── HeadingSection (_H3.2) with 5 children
    │   ├── Heading (H3.7): '### SUTA Dumping Hurts Everyone'
    │   ├── List (L9): '<mistletoe.block_token.List with 1 child line_number=9 loose=False start=None at 0x1040b00b0>'
    │   │   ╰── ListItem (LI9): "'* List immediate after heading\\n'"
    │   │       ╰── Paragraph (P9) of length 29 across 1 children: 'List immediate after heading'
    │   ├── Paragraph (P11) of length 155 across 1 children: 'Employers, employees, and taxpayers...(collapsed)'
    │   ├── Paragraph (P13) of length 14 across 1 

In [11]:
nest_heading_sections(tree)
tree.print()

Tree<'Markdown tree'>
╰── Document (D1): '<mistletoe.block_token.Document with 15 children line_number=1 at 0x103fb2360>'
    ├── Paragraph (P1) of length 45 across 1 children: 'This is the first paragraph with no heading.'
    ╰── HeadingSection (_H1.1) with 5 children
        ├── Heading (H1.3): '# Required Filings and Due Dates'
        ├── Paragraph (P5) of length 46 across 1 children: 'This paragraph is under the first H1 heading.'
        ├── HeadingSection (_H3.2) with 5 children
        │   ├── Heading (H3.7): '### SUTA Dumping Hurts Everyone'
        │   ├── List (L9): '<mistletoe.block_token.List with 1 child line_number=9 loose=False start=None at 0x1040b00b0>'
        │   │   ╰── ListItem (LI9): "'* List immediate after heading\\n'"
        │   │       ╰── Paragraph (P9) of length 29 across 1 children: 'List immediate after heading'
        │   ├── Paragraph (P11) of length 155 across 1 children: 'Employers, employees, and taxpayers...(collapsed)'
        │   ├── Paragraph 

In [12]:
add_list_and_table_intros(tree)

5

In [13]:
" > ".join(get_raw_parent_headings(tree['P38']))

'Required Filings and Due Dates > Scheduled Events > Orientation'

In [37]:
correct_md = """This is the first paragraph with no heading.

# Required Filings and Due Dates

This paragraph is under the first H1 heading.

### SUTA Dumping Hurts Everyone

* List immediate after heading

Employers, employees, and taxpayers make up the difference in higher taxes, lost jobs, lost profits, lower wages, and higher costs for goods and services.

SUTA dumping:

* Costs the UI trust fund millions of dollars each year. Sublist follows:
  * Adversely affects tax rates for all employers.
  * Creates inequity for compliant employers. [some link](https://edd.ca.gov/en/disability/pfl_claim_process/)
* Eliminates the incentive for employers to avoid layoffs.
  * Compromises the integrity of the UI system.

### [SUTA Dumping Schemes](https://edd.ca.gov/en/payroll_taxes/suta_dumping/#collapse-2a82e068-3b29-4473-af7e-de9ea6961277)

These schemes are meant to unlawfully lower an employer’s UI tax rate. Employers should know about these schemes and their potential legal ramifications.

## Scheduled Events

### Orientation

**Career Center Orientation**

| Location | Date/Time | Other Information        |
| -------- | --------- | ------------------------ |
| Virtual  | Friday    | This workshop will ...   |
| Virtual  | Fri       | This workshop is for ... |
| Virtual  | F         | Staff will conduct ...   |

This is the last paragraph. In markdown and html, we can't associate this under the first H1 heading.
"""

"\nThis is the first paragraph with no heading.\n\n# Required Filings and Due Dates\n\nThis paragraph is under the first H1 heading.\n\n### SUTA Dumping Hurts Everyone\n\n* List immediate after heading\n\nEmployers, employees, and taxpayers make up the difference in higher taxes, lost jobs, lost profits, lower wages, and higher costs for goods and services.\n\nSUTA dumping:\n\n* Costs the UI trust fund millions of dollars each year. Sublist follows:\n  * Adversely affects tax rates for all employers.\n  * Creates inequity for compliant employers. [some link](https://edd.ca.gov/en/disability/pfl_claim_process/)\n* Eliminates the incentive for employers to avoid layoffs.\n  * Compromises the integrity of the UI system.\n\n### [SUTA Dumping Schemes](https://edd.ca.gov/en/payroll_taxes/suta_dumping/#collapse-2a82e068-3b29-4473-af7e-de9ea6961277)\n\nThese schemes are meant to unlawfully lower an employer’s UI tax rate. Employers should know about these schemes and their potential legal rami

In [38]:
md = render_tree(tree)
print(md)


This is the first paragraph with no heading.

# Required Filings and Due Dates

This paragraph is under the first H1 heading.

### SUTA Dumping Hurts Everyone

* List immediate after heading

Employers, employees, and taxpayers make up the difference in higher taxes, lost jobs, lost profits, lower wages, and higher costs for goods and services.

SUTA dumping:

* Costs the UI trust fund millions of dollars each year. Sublist follows:
  * Adversely affects tax rates for all employers.
  * Creates inequity for compliant employers. [some link](https://edd.ca.gov/en/disability/pfl_claim_process/)
* Eliminates the incentive for employers to avoid layoffs.
  * Compromises the integrity of the UI system.

### [SUTA Dumping Schemes](https://edd.ca.gov/en/payroll_taxes/suta_dumping/#collapse-2a82e068-3b29-4473-af7e-de9ea6961277)

These schemes are meant to unlawfully lower an employer’s UI tax rate. Employers should know about these schemes and their potential legal ramifications.

## Scheduled

In [39]:
from difflib import SequenceMatcher
from difflib import unified_diff

seq_match = SequenceMatcher(None, correct_md, md)
ratio = seq_match.ratio()
print(ratio)

diff = unified_diff(correct_md, md, lineterm='')
print('\n'.join(list(diff)))

correct_md == md

1.0



True

In [15]:
tree['TR33'].data.token.__dict__

{'row_align': [None, None, None],
 'line_number': 33,
 '_children': [<mistletoe.block_token.TableCell with 1 child line_number=33 align=None at 0x1040b0d70>,
  <mistletoe.block_token.TableCell with 1 child line_number=33 align=None at 0x1040b0e00>,
  <mistletoe.block_token.TableCell with 1 child line_number=33 align=None at 0x1040b0e60>],
 '_parent': <mistletoe.block_token.Table with 3 children line_number=31 column_align=[None, None, None] at 0x1040b0a70>,
 'header': <mistletoe.block_token.TableRow with 3 children line_number=31 row_align=[None, None, None] at 0x1040b0b00>,
 'type': 'TableRow'}

In [16]:
def chunk_nodes(node: Node, memo: dict) -> None:
    # assert node.data.is_block_token(), f"Expecting block-token, not {node.data.token}"
    md_str = node.data.render()
    if len(md_str) < CHAR_LIMIT:
        print(
            f"YAY: Chunked {node.data.id} with len {len(md_str)}: {md_str[:20]!r}...{md_str[-10:]!r}"
        )
        memo[node.data.id].append(md_str)
        # Don't visit child nodes
        return

    print(f"?? Chunking {node.data.id} with {len(md_str)}")
    chunked_list = []
    # First iteration: chunk heading sections
    childs = (n.data.type for n in node.children)
    print("Childs", list(childs))
    for n in (n for n in node.children if n.data.type == "HeadingSection"):
        chunk_nodes(n, memo)
        if n.data.id in memo or n.data.id + ".0" in memo:  # if chunked
            chunked_list.append(n)

    # Try chunking again but with shortened heading sections
    md_str_list = []
    for n in node.children:
        if n in chunked_list:
            md_str_list.append(summarize(n))
            continue
        else:
            md_str = n.data.render()
            if len(md_str) > CHAR_LIMIT:
                raise AssertionError(
                    f"Child section too long for {n.data.type} {n.data.id} with {len(md_str)}"
                )
            else:
                md_str_list.append(md_str)

    # If fit in single chunk
    md_str = "\n".join(md_str_list)
    if len(md_str) <= CHAR_LIMIT:
        print(
            f"YAY: Chunked {node.data.id} with len {len(md_str)}: {md_str[:20]!r}...{md_str[-10:]!r}"
        )
        memo[node.data.id].append(md_str)
        return

    # Split into chunks
    print(f"Retrying chunking {node.data.id} with {len(md_str)}")
    counter = itertools.count()
    md_str = ""
    for md in md_str_list:
        if len(md_str) > CHAR_LIMIT:
            raise AssertionError(f"Too long {len(md_str)}: {md_str}")
        next_md_str = md_str + "\n" + md
        if len(next_md_str) > CHAR_LIMIT:
            memo_key = node.data.id + f".{next(counter)}"
            memo[memo_key].append(md_str)
            print(f"Chunked {memo_key} with len {len(md_str)}")
            md_str = md
    if md_str:  # Last chunk
        memo_key = node.data.id + f".{next(counter)}"
        memo[memo_key].append(md_str)
        print(f"Chunked {memo_key} with len {len(md_str)}")

    if next(counter) == 0:
        print(f"!! Overfilling chunk for {node.data.id}")  # FIXME
        md_str = n.data.render()
        memo[node.data.id].append(md_str)


In [17]:
from collections import defaultdict
from pprint import pprint
CHAR_LIMIT = 500

heading_sections = [n for n in tree.iterator(method=IterMethod.POST_ORDER) if n.data.type == "HeadingSection"]

chunking_memo = defaultdict(list)
for c in heading_sections: #tree.children:
    chunk_nodes(c, memo=chunking_memo)
pprint(chunking_memo) #, width=120)

?? Chunking _H3.2 with 589
Childs ['Heading', 'List', 'Paragraph', 'Paragraph', 'List']
Retrying chunking _H3.2 with 585
!! Overfilling chunk for _H3.2
YAY: Chunked _H3.3 with len 280: '### [SUTA Dumping Sc'...'ications.\n'
YAY: Chunked _H3.5 with len 414: '### Orientation\n\n\n**'...' heading.\n'
YAY: Chunked _H2.4 with len 436: '## Scheduled Events\n'...' heading.\n'
?? Chunking _H1.1 with 1392
Childs ['Heading', 'Paragraph', 'HeadingSection', 'HeadingSection', 'HeadingSection']
?? Chunking _H3.2 with 589
Childs ['Heading', 'List', 'Paragraph', 'Paragraph', 'List']
Retrying chunking _H3.2 with 585
!! Overfilling chunk for _H3.2
YAY: Chunked _H3.3 with len 280: '### [SUTA Dumping Sc'...'ications.\n'
YAY: Chunked _H2.4 with len 436: '## Scheduled Events\n'...' heading.\n'
YAY: Chunked _H1.1 with len 213: '# Required Filings a'...'UMMARIZED)'
defaultdict(<class 'list'>,
            {'_H1.1': ['# Required Filings and Due Dates\n'
                       '\n'
                       'This p

In [18]:
tree.print()

Tree<'Markdown tree'>
╰── Document (D1): '<mistletoe.block_token.Document with 15 children line_number=1 at 0x103fb2360>'
    ├── Paragraph (P1) of length 45 across 1 children: 'This is the first paragraph with no heading.'
    ╰── HeadingSection (_H1.1) with 5 children
        ├── Heading (H1.3): '# Required Filings and Due Dates'
        ├── Paragraph (P5) of length 46 across 1 children: 'This paragraph is under the first H1 heading.'
        ├── HeadingSection (_H3.2) with 5 children
        │   ├── Heading (H3.7): '### SUTA Dumping Hurts Everyone'
        │   ├── List (L9): '<mistletoe.block_token.List with 1 child line_number=9 loose=False start=None at 0x1040b00b0>'
        │   │   ╰── ListItem (LI9): "'* List immediate after heading\\n'"
        │   │       ╰── Paragraph (P9) of length 29 across 1 children: 'List immediate after heading'
        │   ├── Paragraph (P11) of length 155 across 1 children: 'Employers, employees, and taxpayers...(collapsed)'
        │   ├── Paragraph 

In [19]:
print(tree.children[0].data.render())

This is the first paragraph with no heading.
# Required Filings and Due Dates
This paragraph is under the first H1 heading.
### SUTA Dumping Hurts Everyone
* List immediate after heading
Employers, employees, and taxpayers make up the difference in higher taxes, lost jobs, lost profits, lower wages, and higher costs for goods and services.
SUTA dumping:
* Costs the UI trust fund millions of dollars each year. Sublist follows:
    * Adversely affects tax rates for all employers.
    * Creates inequity for compliant employers. [some link](https://edd.ca.gov/en/disability/pfl_claim_process/)
* Eliminates the incentive for employers to avoid layoffs.
    * Compromises the integrity of the UI system.
### [SUTA Dumping Schemes](https://edd.ca.gov/en/payroll_taxes/suta_dumping/#collapse-2a82e068-3b29-4473-af7e-de9ea6961277)
These schemes are meant to unlawfully lower an employer’s UI tax rate. Employers should know about these schemes and their potential legal ramifications.
## Scheduled Even

In [20]:
tree.children[0].data.token.__dict__

{'footnotes': {},
 'line_number': 1,
 '_children': [<mistletoe.block_token.Paragraph with 1 child line_number=1 at 0x103ffc4a0>,
  <mistletoe.block_token.Heading with 1 child line_number=3 level=1 at 0x10403bdd0>,
  <mistletoe.block_token.Paragraph with 1 child line_number=5 at 0x103fedca0>,
  <mistletoe.block_token.Heading with 1 child line_number=7 level=3 at 0x1040b0050>,
  <mistletoe.block_token.List with 1 child line_number=9 loose=False start=None at 0x1040b00b0>,
  <mistletoe.block_token.Paragraph with 1 child line_number=11 at 0x1040b00e0>,
  <mistletoe.block_token.Paragraph with 1 child line_number=13 at 0x1040b02c0>,
  <mistletoe.block_token.List with 2 children line_number=15 loose=True start=None at 0x1040b0320>,
  <mistletoe.block_token.Heading with 1 child line_number=22 level=3 at 0x1040b0350>,
  <mistletoe.block_token.Paragraph with 1 child line_number=24 at 0x1040b0740>,
  <mistletoe.block_token.Heading with 1 child line_number=26 level=2 at 0x1040b07d0>,
  <mistletoe.

In [21]:
a =b =1
(a,b)

(1, 1)

In [22]:
vars(tree['T31'].data.token)

{'column_align': [None, None, None],
 'header': <mistletoe.block_token.TableRow with 3 children line_number=31 row_align=[None, None, None] at 0x1040b0b00>,
 '_children': [<mistletoe.block_token.TableRow with 3 children line_number=33 row_align=[None, None, None] at 0x1040b09b0>,
  <mistletoe.block_token.TableRow with 3 children line_number=34 row_align=[None, None, None] at 0x1040b0ce0>,
  <mistletoe.block_token.TableRow with 3 children line_number=35 row_align=[None, None, None] at 0x1040b0d10>],
 'line_number': 31,
 '_parent': <mistletoe.block_token.Document with 15 children line_number=1 at 0x103fb2360>,
 'type': 'Table',
 'col_widths': [8, 9, 24],
 'sep_line': ['--------', '---------', '------------------------']}

In [23]:
tree['T31'].data.token.header.children[0].children[0].content

'Location'

In [24]:
vars(tree.children[0].children[0].data.token)

{'_children': [<mistletoe.span_token.RawText content='This is the first paragraph wi'...+14 at 0x104038c50>],
 'line_number': 1,
 '_parent': <mistletoe.block_token.Document with 15 children line_number=1 at 0x103fb2360>,
 'type': 'Paragraph'}